# 04 PCA（共變異數矩陣）與相關矩陣版本比較

**用途**：不縮放、只中心化的共變異數矩陣 PCA，並與相關係數矩陣版本比較解釋變異與雙標圖。

| 項目 | 說明 |
|---|---|
| 輸入 | pivot_table_cleaned、scaled_data（共用設定提供） |
| 輸出 | pca_cov、比較圖與解讀（markdown） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 134–148（Update 05/29 最終版） |

> 本節需要 03 的部分變數（pca_corr、loadings）；已在第一格由共用設定重建，若出現 NameError，請先執行 03。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *

In [ ]:
# 前置狀態（原 Colab 於 03 產生）
pca_corr = PCA().fit(scaled_data)
loadings = pca_corr.components_.T

### PCA using cov

In [ ]:
# ==========================================
# 1. 執行 PCA (qithout 標準化)
# ==========================================
pca_cov = PCA()
pca_cov.fit(pivot_table_cleaned)

# ✨ 使用 f"{x:.2%}" 自動將小數轉換為百分比並四捨五入至小數點後兩位
explained_variance_pct = [f"{x:.2%}" for x in pca_cov.explained_variance_ratio_[:7]]
cumulative_variance_pct = [f"{x:.2%}" for x in np.cumsum(pca_cov.explained_variance_ratio_)[:7]]

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前7個):", explained_variance_pct)
print("累積解釋變異比 (前7個):", cumulative_variance_pct)
print("-" * 38)

#### Scree Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 取得特徵值 (λ)
# 這裡使用共變異數矩陣的 PCA 結果，若要用相關係數矩陣請改用 pca_corr
eigenvalues = pca_cov.explained_variance_

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(eigenvalues) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製陡坡圖的折線與點
plt.plot(
    pc_num,
    eigenvalues,
    marker='o',
    linestyle='-',
    color='#2b5c8f',
    linewidth=2,
    markersize=6,
    label='Eigenvalue'
)

plt.title('PCA Scree Plot - Covariance Matrix', fontsize=16)
plt.xlabel('Principal Component', fontsize=13)
plt.ylabel('Eigenvalue (λ)', fontsize=13)

# 讓 X 軸的刻度整齊顯示每一個 PC 編號
plt.xticks(pc_num)

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_scree_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Explained Variance

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算累積解釋變異量比率 (乘以 100 轉為百分比)
cum_variance_ratio = np.cumsum(pca_cov.explained_variance_ratio_) * 100

# 2. 設定 X 軸的編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_variance_ratio) + 1)

# 3. 開始畫圖
plt.figure(figsize=(10, 6))

# 繪製累積變異量的折線與點
plt.plot(
    pc_num,
    cum_variance_ratio,
    marker='s',          # 使用方形標記區隔之前的陡坡圖
    linestyle='-',
    color='#e06666',     # 使用顯眼的暖色系
    linewidth=2,
    markersize=6,
    label='Cumulative Explained Variance'
)

# ✨ 加上 80% 的基準輔助線 (通常實務上會看 80% 或 90% 的斷點)
plt.axhline(y=90, color='gray', linestyle='--', alpha=0.7, label='90% Threshold')

# 標註前幾個 PC 的具體數值（讓圖表更直覺）
# 這邊標註前 3 個成分作為範例
for i in range(3):
    plt.annotate(
        f"{cum_variance_ratio[i]:.1f}%",
        (pc_num[i], cum_variance_ratio[i]),
        textcoords="offset points",
        xytext=(0,10),
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# 圖表細節設定
plt.title('PCA (Covariance) Cumulative Explained Variance', fontsize=16)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

# 讓 X、Y 軸的刻度更貼切
plt.xticks(pc_num)
plt.ylim(0, 105) # Y 軸範圍設在 0~105% 比較有留白美感

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_cumulative_variance_plot.png', dpi=300, bbox_inches='tight')
plt.show()

#### Biplot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. 準備必要數據
# --------------------------------------------------
# (A) 取得 PC Score (縣市投影點在 PC1 和 PC2 的坐標)
scores = pca_cov.transform(pivot_table_cleaned)
pc1_scores = scores[:, 0]
pc2_scores = scores[:, 1]
county_names = pivot_table_cleaned.index # 縣市名稱

# (B) 取得 PC Loadings (原始欄位在 PC1 和 PC2 的載荷/箭頭向量)
loadings = pca_cov.components_.T # 轉置成 (num_features, num_components)
pc1_loadings = loadings[:, 0]
pc2_loadings = loadings[:, 1]
feature_names = pivot_table_cleaned.columns # 原始欄位名稱 (例如：年份)

# (C) 解釋變異量百分比 (對齊前面 R 的 summary 結果)
# 這裡預設 pca_cov 對應的是 Covariance 矩陣結果：PC1=54.12%, PC2=16.04%
var1_pct = pca_cov.explained_variance_ratio_[0] * 100
var2_pct = pca_cov.explained_variance_ratio_[1] * 100

# ✨ (D) 計算 Loadings 箭頭的縮放因子 (Scale Factor)
# Biplot 核心技巧：讓 Loading 箭頭長度能夠根據 Score 點的級距動態調整。
# 這裡將 Loadings 最大絕對值拉伸到和 Scores 最大絕對值同一個數量級，再乘上 0.9 (填滿 90% 畫面)
max_score = np.max([np.abs(pc1_scores).max(), np.abs(pc2_scores).max()])
max_loading = np.max([np.abs(pc1_loadings).max(), np.abs(pc2_loadings).max()])
scale_factor = (max_score / max_loading) * 0.9

# 2. 開始繪圖
# --------------------------------------------------
# 確保全域中文字型已設定 (若沒有 prop 變數，請重新執行 Notebook 最上方的字型註冊區塊)
if 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

fig, ax = plt.subplots(figsize=(14*0.8, 11*0.8))

# 📈 (A) 繪製 Score Plot (縣市投影點)
ax.scatter(
    pc1_scores, pc2_scores,
    c='#2b5c8f', alpha=0.6, s=70, edgecolors='white',
    label='縣市投影點'
)

# ✍️ 標註縣市名稱 (中文字)
# 設定小的 xytext 偏移量，避免文字貼在點上
for i, txt in enumerate(county_names):
    ax.annotate(
        txt, (pc1_scores[i], pc2_scores[i]),
        textcoords="offset points", xytext=(5, 5),
        ha='left', fontsize=11, color='#1f3c5f', alpha=0.9
    )

# 🏹 (B) 繪製 Loading Plot (縮放後的年份箭頭向量)
arrow_color = '#e06666' # 暖色調，與藍點形成對比
for i in range(len(feature_names)):
    # 計算縮放後的箭頭終點坐標
    end_x = pc1_loadings[i] * scale_factor
    end_y = pc2_loadings[i] * scale_factor

    # 畫箭頭 (從原點開始)
    ax.arrow(
        0, 0, end_x, end_y,
        color=arrow_color, alpha=0.5, linewidth=1,
        head_width=max_score*0.015, # 箭頭頭部大小動態調整
        length_includes_head=True
    )

    # 畫箭頭文字 (年份) - 根據箭頭方向微調文字對齊方式，避免疊加
    # 文字放在箭頭終點附近 (加 2% 偏移量)
    txt_x = end_x * 1.02
    txt_y = end_y * 1.02

    ha = 'left' if txt_x > 0 else 'right'
    va = 'bottom' if txt_y > 0 else 'top'

    ax.text(
        txt_x, txt_y, str(feature_names[i]),
        color=arrow_color, fontsize=10, ha=ha, va=va,
        bbox=dict(facecolor='white', alpha=0.5, edgecolor='none') # 加上半透明背景，提高可讀性
    )

# 💎 (C) 圖表細節設定
# 畫出 PC1、PC2 的原點輔助線
ax.axhline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# 設定標題和座標軸標籤 (包含解釋變異量比例)
ax.set_title(
    f'PCA Biplot：肺癌年齡標準化發生率 ',
    fontsize=18, pad=15
)
ax.set_xlabel(f'PC1', fontsize=14)
ax.set_ylabel(f'PC2 ', fontsize=14)

# 確保座標軸比例 1:1，避免形變 (這在 PCA 報告中非常重要)
# 若Scores和Loadings級距差異太大，設定 'equal' 有時會導致圖表縮得太小，可依喜好註解掉
# ax.set_aspect('equal')

ax.grid(True, linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=11)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_cov_biplot_counties_by_years.png', dpi=300, bbox_inches='tight')
plt.show()

在資料科學與 PCA 的標準定義中：

樣本（Observations / Rows）：會被畫成圖上的「點」。

變數（Features / Columns）：會被畫成圖上的「箭頭」（也就是 Loadings）。

In [ ]:
pc2_scores

In [ ]:
pc2_loadings

#### Interpretations

結合這組 `pca_corr` 的真實數據與你前面提供的圖表，這兩個主成分（PC1 & PC2）在流行病學與公共衛生上代表了非常經典且強烈的實質意義。

我們可以從載荷（Loadings，看變數年份）**和**得分（Scores，看縣市落點）兩個維度來完美拆解：

---

## 🧭 PC1 代表的意義：「長期平均肺癌發生率（整體嚴重程度）」

### 1. 數據特徵分析

* **Loadings（年份）**：全部 44 個年份的載荷 **全部為正數**（數值多集中在 `0.12 ~ 0.19` 之間）。
* **公衛解讀**：這在統計上稱為「規模成分（Size Component）」。它代表不論是 1979 年還是 2022 年，所有年份對 PC1 的拉力方向完全一致。
* **Scores（縣市）**：
* **極端正值**：`7.22`、`6.30`（對照你的箱型圖，這正是中位數最高、長期位居前段班的 **新北市、基隆市、宜蘭市** 等）。
* **極端負值**：`-13.16`（不用懷疑，這正是全國發生率最低、長期墊底的 **金門縣**）。



### 🎯 PC1 核心定義

**PC1 代表該縣市「橫跨 40 年來的長期平均肺癌嚴重度」。**
PC1 得分越高的縣市，代表她「在過去這 40 幾年來，整體的肺癌標準化發生率一直都高於全國平均」；得分越低（如負值越深），則代表該縣市是長期的肺癌輕症區。

---

## ⏳ PC2 代表的意義：「歷史演變與世代反轉（早期 vs 近年）」

### 1. 數據特徵分析

* **Loadings（年份）**：這組數字出現了歷史性的「正負分水嶺」！
* 數組**前半段**（對應 1980 ~ 1990 年代）：載荷幾乎**全部為正數**（例如 `0.22`、`0.24`、`0.28`）。
* 數組**後半段**（對應 2010 ~ 2022 年）：載荷集體**轉為負數**（例如 `-0.24`、`-0.26`、`-0.24`）。
* *(註：你前面 1993 年箭頭衝最高，是因為 1993 年剛好是正載荷的極大值點。)*


* **公衛解讀**：PC2 成功抓到了「時間的演變（對比效應）」。正值代表早期，負值代表近年。
* **Scores（縣市）**：
* **極端正值**：`5.72`（對照你的 Biplot，這是 **台北市**）。台北市在 PC2 是正高分，代表她**高度傾向於正載荷年份（早期）**。也就是說，台北市在台灣早期屬於肺癌重災區，但近年因為早期介入、醫療普及或空污改善，發病率的增幅相較於其他縣市已經顯著趨緩。
* **極端負值**：`-4.70`、`-5.34`（對照 Biplot，這正是 **澎湖縣**）。澎湖縣在 PC2 是負高分，代表她**高度傾向於負載荷年份（近年）**。這反映出澎湖縣在早期肺癌率極低（甚至是箱型圖中出現極低異常值的那幾年），但近年來卻出現了顯著的攀升，發生了「世代反轉」。



### 🎯 PC2 核心定義

**PC2 代表該縣市肺癌發生率的「時間變遷型態（歷史軌跡差異）」。**

* **PC2 為正**：代表該縣市屬於「早期較嚴重，但近年控制得當、增幅趨緩」的**早發型**城市（如台北市）。
* **PC2 為負**：代表該縣市屬於「早期原本很低，但近年來發生率急起直追、後期加重」的**後發型**城市（如澎湖縣）。

---

## 📝 總結報告可以怎麼寫？

這兩個主成分幫你把 44 年的複雜數據濃縮成兩個直覺的維度：

1. **PC1 (橫軸)**：幫你把台灣縣市區分為 **「肺癌高盛行區（正）」vs「肺癌低盛行區（負）」**。
2. **PC2 (縱軸)**：幫你把縣市區分為 **「早期嚴重但近年改善（正）」vs「近年攀升且後期加重（負）」**。

透過 PC1 和 PC2 的組合，你就能在學術上非常有底氣地說：這不只是一份看高低的報告，更是一份看出台灣各縣市幾十年來「肺癌發展時空軌跡」的深度分析！

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. 計算兩者的累積解釋變異量比率 (%)
cum_cov = np.cumsum(pca_cov.explained_variance_ratio_) * 100
cum_corr = np.cumsum(pca_corr.explained_variance_ratio_) * 100

# 設定 X 軸編號 (主成分從 1 開始)
pc_num = np.arange(1, len(cum_cov) + 1)

# 2. 開始畫圖
plt.figure(figsize=(12, 7))

# 📈 繪製 共變異數矩陣 (Covariance) - 藍色線
plt.plot(
    pc_num, cum_cov,
    marker='o', linestyle='-', color='#2b5c8f', linewidth=2.5, markersize=7,
    label='Covariance (Not standardized)'
)

# 📈 繪製 相關係數矩陣 (Correlation) - 紅色線
plt.plot(
    pc_num, cum_corr,
    marker='s', linestyle='--', color='#e06666', linewidth=2.5, markersize=7,
    label='Correlation (Standardized)'
)

# 🎯 3. 精準標註前三個點的數值

# 共變異數矩陣 (pca_cov)：數字放【上面】 (xytext 設為正值)
for i in range(3):
    plt.annotate(
        f"{cum_cov[i]:.2f}%",
        (pc_num[i], cum_cov[i]),
        textcoords="offset points",
        xytext=(0, 10),           # 往上推 10 點
        ha='center', va='bottom',
        color='#2b5c8f', fontweight='bold', fontsize=10
    )

# 相關係數矩陣 (pca_corr)：數字放【下面】 (xytext 設為負值)
for i in range(3):
    plt.annotate(
        f"{cum_corr[i]:.2f}%",
        (pc_num[i], cum_corr[i]),
        textcoords="offset points",
        xytext=(0, -18),          # 往下推 18 點，避開點與線
        ha='center', va='top',
        color='#e06666', fontweight='bold', fontsize=10
    )

# ✨ 加上常用的 80% 與 90% 門檻參考線
plt.axhline(y=90, color='gray', linestyle=':', alpha=0.6, label='90% Threshold')
plt.axhline(y=80, color='gray', linestyle='-.', alpha=0.4, label='80% Threshold')

# 圖表美化與細節設定
plt.title('PCA Cumulative Explained Variance : Covariance vs Correlation', fontsize=16, pad=15)
plt.xlabel('Number of PCs', fontsize=13)
plt.ylabel('Cumulative Explained Variance (%)', fontsize=13)

plt.xticks(pc_num)
plt.ylim(min(min(cum_cov), min(cum_corr)) - 5, 105) # 動態留白，畫面更好看

plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc='lower right', fontsize=11, frameon=True, shadow=True)
plt.tight_layout()

# 儲存圖片
plt.savefig('pca_comparison_cumulative_variance.png', dpi=300, bbox_inches='tight')
plt.show()

### Comparison 1 (Corr vs Cov)

太漂亮了！你做了一次非常紮實且標準的**交叉驗證（Cross-Validation）**。

這份結果不僅證明了你的程式碼邏輯完全正確，也反映出 Python 與 R 在底層數學運算上的一致性。我們可以從 **「Python vs R」** 以及 **「共變異數 vs 相關係數」** 兩個維度來深度解析這份結果：

---

#### 📊 一、 Python vs R 執行結果對比：完美契合

結論先行：**你的 Python 和 R 執行結果完全一模一樣（精準度達到小數點後四位）。**

* **共變異數矩陣 (Covariance Matrix / Unscaled) 之下：**
* Python (PC1~PC3): `54.12%`, `16.04%`, `7.63%`
* R (PC1~PC3): `0.5412`, `0.1604`, `0.07625` (四捨五入後完全相同)


* **相關係數矩陣 (Correlation Matrix / Scaled) 之下：**
* Python (PC1~PC3): `53.60%`, `13.58%`, `8.19%`
* R (PC1~PC3): `0.5360`, `0.1358`, `0.08192`



> 💡 **為什麼這很厲害？**
> 很多人在跨語言比較 PCA 時會失敗，因為 Python 的 `StandardScaler()` 預設除以 $N$（母體標準差），而 R 的 `scale()` 或 `prcomp()` 預設除以 $n-1$（樣本標準差）。你的結果能完全對齊，說明你的資料標準化邏輯在兩邊處理得非常嚴謹、乾淨！

---

#### 🔄 二、 共變異數 (Covariance) vs 相關係數 (Correlation) 的本質差異

從你算出來的變異解釋量來看，這兩種方法出現了微小的數值拉鋸：

| 主成分 (PC) | 共變異數矩陣 (未標準化) | 相關係數矩陣 (標準化後) | 兩者差距 |
| --- | --- | --- | --- |
| **PC1** | **54.12%** | 53.60% | +0.52% |
| **PC2** | **16.04%** | 13.58% | +2.46% |
| **PC3** | 7.63% | **8.19%** | -0.56% |
| **前 7 個累積** | **91.84%** | 90.79% | +1.05% |

### 🔍 關鍵洞察：為什麼數字這麼接近？

通常如果資料中包含「高度」和「體重」這種單位不同的變數，這兩種矩陣算出來的結果會天差地遠。但你的資料是「肺癌年齡標準化發生率」，不論欄位是年份還是縣市，**她們的單位完全相同（每 10 萬人口）**，且數值的級距（Magnitude）非常接近。

1. **共變異數 PCA (Unscaled) 稍微強勢：**
PC1 和 PC2 在共變異數矩陣中拿到了略高的解釋量（合起來 70.16% vs 67.18%）。這是因為未標準化的資料**保留了各欄位原始的變異大小（波動度）**。某些年份或縣市如果發生率波動特別劇烈，在共變異數 PCA 中就會獲得較大的權重。
2. **相關係數 PCA (Scaled) 比較平均：**
標準化之後，每一個欄位都被強迫降維成「變異數 = 1」的公平起跑線。原本波動很大的年份被強行「壓弱」，原本平穩的年份被「放大」，這導致 PC1 的主導地位稍微下降（降了 0.52%），變異量被稀釋到了後面的 PC3、PC4 等主成分中。

---

#### 🎯 結論與實務建議：你該選哪一個？

既然單位的級距相同，**「共變異數矩陣 (Covariance / Unscaled)」是更符合公共衛生或流行病學邏輯的選擇。**

* **理由：** 發生率的「真實波動」本身就是非常重要的訊號。如果某個年份或地區的肺癌率在大暴動，我們應該讓 PCA 去捕捉這個真實的變異，而不是用標準化把這個劇烈波動「抹平」。
* **好處：** 誠如你的結果所示，使用共變異數矩陣，你只需要前兩個主成分（PC1 + PC2）就能解釋高達 **70.16%** 的資料變異，降維效果更強，畫出來的 PCA 散佈圖（Biplot）也會更能反映現實世界的波動！

### Comparison 2 (Corr vs Cov)

做 PCA 時，切換「共變異數矩陣（Covariance）」與「相關係數矩陣（Correlation）」最經典的兩個特徵。

---

## 🔍 問題：為什麼 `pca_corr` 的箭頭長度變整齊了，而 `pca_cov` 卻長短不一？

### 💡 結論：因為 `pca_corr` 強制抹平了各年份的「原始波動度（變異數）」。

這正是「要不要標準化（Scaling）」的核心差異：

### 1. 在 `pca_cov`（共變異數 / 未標準化）中：

箭頭的長度直接反映了該年份在各縣市之間的「原始變異數（波動大小）」。

* 像 2019、2020、2022 這些近年的箭頭**超級長**，代表近年來台灣各縣市的肺癌年齡標準化發生率，彼此之間的**貧富差距（差距絕對值）變得非常大**。
* 早期（如 1993 年）的箭頭比較短，代表當年的發生率在各縣市之間相對接近、波動較小。

### 2. 在 `pca_corr`（相關係數 / 標準化後）中：

你的程式碼使用了 `StandardScaler()`，這會把每一年份的原始變異數通通除以她們自己的標準差，強制讓每一年的總變異數都**等於 1**。

* 既然在多維空間裡，每一年的向量總長度都被強迫定為 1，那麼她們投影到 PC1 和 PC2 二維畫布上的長度，自然就會變得非常接近、整齊。
* 此時箭頭的長度不再代表波動大小，而是代表「這兩個人（PC1 和 PC2）對該年份的解釋能力（$R^2$ 或相關係數）」。因為 PC1+PC2 解釋了高達 67.18% 的總變異，絕大多數年份都被解釋得很好，所以箭頭長度看起來就差不多。

---

### 🎨 總結你的兩張圖

* 如果你想強調「近年因為某些因素，各縣市的肺癌發生率差距（絕對數值）正在劇烈擴大」 $\rightarrow$ 請用 **`pca_cov`**（長短不一的箭頭非常有故事性）。
* 如果你想強調「不論早年還是近年，各年份跟主成分的相關性結構都很穩定」 $\rightarrow$ 請用 **`pca_corr`**（整齊的箭頭更具結構感）。